# 45 — Avaliação OOD: 4 corpora externos × todos os modelos

Aplica os modelos treinados (BERT, TF-IDF — binário e multiclasse) sobre quatro corpora externos.

| Dataset | Positivo binário | Níveis avaliados |
|---|---|---|
| **Fake.Br** (Lira et al.) | `metadata_category == 'economia'` (44/7200, 0.6%) | 1 binário, 2 multiclasse, 3 subgrupo por veracidade |
| **FakeRecogna** | URL do portal `economia.uol.com.br` (312/11872, 2.6%) | 1 binário, 2 multiclasse, 3 corte balanceado intra-UOL |
| **PortugueseNewsDataset** (Klaifer/WikiNotícias, PLOS ONE 2024) | `category == 'Economia e negócios'` (1.151/9.135, 12.6%) | 1 binário (partição `full`) |
| **RecognaSumm** (Paiola et al., PROPOR 2024) | `Categoria == 'Economia'` (2.515/27.055, 9.3%) | 1 binário (partição `test`) |

Cada nível emite `result_card.json` em `<DRIVE>/ood_runs/<model_id>_<task>_<dataset>_<level>/`. A célula final agrega cards e roda McNemar pareado por `(domain, level)`.

**Escopo binário-only de PN/RS**: PortugueseNewsDataset e RecognaSumm são avaliados **somente no nível binário** (label de tópico direta, sem mapeamento multiclasse). Os modelos multiclasse rodam apenas em Fake.Br e FakeRecogna.

**Pré-requisitos**:
- Três zips em `<DRIVE>/economy-classifier/`:
  - `colab_ood_data.zip` (Fake.Br + FakeRecogna)
  - `colab_portuguese_news.zip` (PortugueseNewsDataset reconstruído via repo Klaifer)
  - `colab_recognasumm.zip` (RecognaSumm test split, ~33 MB)
- Modelos treinados em `<DRIVE>/economy-classifier/runs/<model_id>_<task>_test_set/model/` (HF dir para BERT, `tfidf_pipeline.joblib` para TF-IDF), gerados pelos NBs 21 / 11 / 12 / 13.

## 0. Verificação de ambiente

In [ ]:
import torch

if not torch.cuda.is_available():
    print("AVISO: GPU nao detectada. BERT vai rodar em CPU (lento). "
          "Para acelerar: Runtime > Change runtime type > GPU.")
    GPU_NAME = "CPU"
    HARDWARE = "Colab-CPU"
else:
    GPU_NAME = torch.cuda.get_device_name(0)
    VRAM_GB = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
    print(f"GPU: {GPU_NAME} ({VRAM_GB} GB VRAM)")
    HARDWARE = f"Colab-{GPU_NAME.split()[-1]}"
print("CUDA:", torch.version.cuda)


AVISO: GPU nao detectada. BERT vai rodar em CPU (lento). Para acelerar: Runtime > Change runtime type > GPU.
CUDA: None


## 1. Bootstrap (Colab + local)

In [ ]:
import subprocess
import sys
import zipfile
from pathlib import Path


def _run(cmd: list[str], description: str) -> None:
    print(f"$ {' '.join(cmd)}")
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        print("STDERR:", result.stderr, file=sys.stderr)
        raise RuntimeError(f"{description} failed with exit code {result.returncode}")


IN_COLAB = "google.colab" in sys.modules
print("Ambiente:", "Google Colab" if IN_COLAB else "Local")
print("Python   :", sys.version.split()[0])

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    REPO_URL = "https://github.com/almeidadm/economy-classifier.git"
    REPO_BRANCH = "main"
    DRIVE_FOLDER = "economy-classifier"

    DRIVE_BASE = Path("/content/drive/MyDrive") / DRIVE_FOLDER
    DRIVE_BASE.mkdir(parents=True, exist_ok=True)
    REPO_DIR = Path("/content/economy-classifier")

    if REPO_DIR.exists():
        _run(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_BRANCH], "git fetch")
        _run(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH], "git checkout")
        _run(["git", "-C", str(REPO_DIR), "reset", "--hard", f"origin/{REPO_BRANCH}"], "git reset")
    else:
        _run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)], "git clone")

    _run(
        [sys.executable, "-m", "pip", "install", "-e", str(REPO_DIR),
         "--upgrade-strategy", "only-if-needed", "-q"],
        "pip install -e .",
    )

    if str(REPO_DIR / "src") not in sys.path:
        sys.path.insert(0, str(REPO_DIR / "src"))

    RUNS_BASE = DRIVE_BASE / "runs"
    OOD_RUNS_BASE = DRIVE_BASE / "ood_runs"
    OOD_DATA_DIR = Path("/content/ood_data")
else:
    REPO_DIR = Path.cwd().parent
    DRIVE_BASE = REPO_DIR / "artifacts"
    RUNS_BASE = DRIVE_BASE / "runs"
    OOD_RUNS_BASE = DRIVE_BASE / "ood_runs"
    OOD_DATA_DIR = REPO_DIR / "ood_data"

OOD_RUNS_BASE.mkdir(parents=True, exist_ok=True)
OOD_DATA_DIR.mkdir(parents=True, exist_ok=True)

print("REPO_DIR     :", REPO_DIR)
print("RUNS_BASE    :", RUNS_BASE)
print("OOD_RUNS_BASE:", OOD_RUNS_BASE)
print("OOD_DATA_DIR :", OOD_DATA_DIR)


Ambiente: Google Colab
Python   : 3.12.13
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
$ git -C /content/economy-classifier fetch origin main
$ git -C /content/economy-classifier checkout main
Your branch is up to date with 'origin/main'.

$ git -C /content/economy-classifier reset --hard origin/main
HEAD is now at a602898 feat(ood-eval): scripts + notebook para avaliacao OOD em Fake.Br + FakeRecogna

$ /usr/bin/python3 -m pip install -e /content/economy-classifier --upgrade-strategy only-if-needed -q
REPO_DIR     : /content/economy-classifier
RUNS_BASE    : /content/drive/MyDrive/economy-classifier/runs
OOD_RUNS_BASE: /content/drive/MyDrive/economy-classifier/ood_runs
OOD_DATA_DIR : /content/ood_data


## 2. Imports dos scripts de avaliação

In [ ]:
# scripts/ nao e um pacote Python — adicionamos manualmente ao path
SCRIPTS_DIR = REPO_DIR / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

import evaluate_fake_br as fb
import evaluate_fake_recogna as fr
import evaluate_portuguese_news as pn
import evaluate_recognasumm as rs
from economy_classifier.project import compute_artifact_size_mb
from economy_classifier.evaluation import compute_mcnemar_pairwise

FAKE_BR_ROOT = OOD_DATA_DIR / "fake_br"
FAKE_RECOGNA_ROOT = OOD_DATA_DIR / "fake_recogna"
PORTUGUESE_NEWS_ROOT = OOD_DATA_DIR / "portuguese_news_wikinotices"
RECOGNASUMM_ROOT = OOD_DATA_DIR / "recognasumm"

print("FAKE_BR_ROOT        :", FAKE_BR_ROOT)
print("FAKE_RECOGNA_ROOT   :", FAKE_RECOGNA_ROOT)
print("PORTUGUESE_NEWS_ROOT:", PORTUGUESE_NEWS_ROOT)
print("RECOGNASUMM_ROOT    :", RECOGNASUMM_ROOT)

## 3. Carregamento dos datasets OOD

**Layout esperado** dentro de cada zip (três zips separados):

```
colab_ood_data.zip
  fake_br/
    full_texts/
      fake/<id>.txt
      true/<id>.txt
      fake-meta-information/<id>-meta.txt
      true-meta-information/<id>-meta.txt
  fake_recogna/
    FakeRecogna_*.xlsx

colab_portuguese_news.zip
  portuguese_news_wikinotices/
    wikinews_categories.json
    wikinews_train.json
    wikinews_test.json
    split_ids.csv

colab_recognasumm.zip
  recognasumm/
    test.jsonl
```

**Para gerar os zips localmente** (uma vez, antes do primeiro upload):

```bash
# Fake.Br + FakeRecogna (corpora do repo fn-dataset-eda vizinho)
cd ~/Documentos/repositorios/fn-dataset-eda/data/raw
zip -r ~/Documentos/repositorios/economy-classifier/colab_ood_data.zip fake_br fake_recogna

# PortugueseNewsDataset + RecognaSumm (corpora reconstruidos em data/ do proprio repo)
cd ~/Documentos/repositorios/economy-classifier
uv run python scripts/colab_pack_portuguese_news.py
uv run python scripts/colab_pack_recognasumm.py

# Suba os 3 zips para <DRIVE>/economy-classifier/
```

Em execução **local**, a célula abaixo cria links simbólicos para `fn-dataset-eda/data/raw/` (FB+FR) e para `data/portuguese_news_wikinotices/` + `data/recognasumm/` (PN+RS, reconstruídos no próprio repo).

In [ ]:
PN_PARTITION = "full"   # 9135 docs; troque para "test" (914) para reproduzir paper Klaifer
RS_PARTITION = "test"   # 27.055 docs, o unico no colab_recognasumm.zip


def _extract_zip(zip_path: Path, expected_root: Path) -> None:
    if expected_root.exists():
        return
    assert zip_path.exists(), (
        f"Falta {zip_path}. Veja a secao 3 acima para gerar o zip e fazer upload."
    )
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(OOD_DATA_DIR)
    print(f"Extraido {zip_path.name} -> {OOD_DATA_DIR}")


if IN_COLAB:
    _extract_zip(DRIVE_BASE / "colab_ood_data.zip", FAKE_BR_ROOT)
    _extract_zip(DRIVE_BASE / "colab_ood_data.zip", FAKE_RECOGNA_ROOT)
    _extract_zip(DRIVE_BASE / "colab_portuguese_news.zip", PORTUGUESE_NEWS_ROOT)
    _extract_zip(DRIVE_BASE / "colab_recognasumm.zip", RECOGNASUMM_ROOT)
else:
    LOCAL_FN_REPO = Path("/home/diacrono/Documentos/repositorios/fn-dataset-eda/data/raw")
    if not FAKE_BR_ROOT.exists() and (LOCAL_FN_REPO / "fake_br").exists():
        FAKE_BR_ROOT.symlink_to(LOCAL_FN_REPO / "fake_br")
    if not FAKE_RECOGNA_ROOT.exists() and (LOCAL_FN_REPO / "fake_recogna").exists():
        FAKE_RECOGNA_ROOT.symlink_to(LOCAL_FN_REPO / "fake_recogna")
    LOCAL_PN = REPO_DIR / "data" / "portuguese_news_wikinotices"
    LOCAL_RS = REPO_DIR / "data" / "recognasumm"
    if not PORTUGUESE_NEWS_ROOT.exists() and LOCAL_PN.exists():
        PORTUGUESE_NEWS_ROOT.symlink_to(LOCAL_PN)
    if not RECOGNASUMM_ROOT.exists() and LOCAL_RS.exists():
        RECOGNASUMM_ROOT.symlink_to(LOCAL_RS)

assert FAKE_BR_ROOT.exists(), f"Fake.Br ausente em {FAKE_BR_ROOT}"
assert FAKE_RECOGNA_ROOT.exists(), f"FakeRecogna ausente em {FAKE_RECOGNA_ROOT}"
assert PORTUGUESE_NEWS_ROOT.exists(), f"PortugueseNewsDataset ausente em {PORTUGUESE_NEWS_ROOT}"
assert RECOGNASUMM_ROOT.exists(), f"RecognaSumm ausente em {RECOGNASUMM_ROOT}"

print("Carregando Fake.Br...")
fb_df = fb.load_fake_br(FAKE_BR_ROOT)
print(f"  {len(fb_df)} docs | economia={int((fb_df.y_true_binary==1).sum())}")

print("Carregando FakeRecogna...")
fr_df = fr.load_fake_recogna(FAKE_RECOGNA_ROOT)
print(f"  {len(fr_df)} docs | economia.uol={int((fr_df.y_true_binary==1).sum())}")

print(f"Carregando PortugueseNewsDataset (partition={PN_PARTITION})...")
pn_df = pn.load_portuguese_news(PORTUGUESE_NEWS_ROOT, partition=PN_PARTITION)
print(f"  {len(pn_df)} docs | Economia e negocios={int((pn_df.y_true_binary==1).sum())}")

print(f"Carregando RecognaSumm (partition={RS_PARTITION})...")
rs_df = rs.load_recognasumm(RECOGNASUMM_ROOT, partition=RS_PARTITION)
print(f"  {len(rs_df)} docs | Economia={int((rs_df.y_true_binary==1).sum())}")

# Caches reaproveitados em todos os modelos
fb_texts = fb_df["text"].fillna("").tolist()
fr_texts = fr_df["text"].fillna("").tolist()
pn_texts = pn_df["text"].fillna("").tolist()
rs_texts = rs_df["text"].fillna("").tolist()

## 4. Descoberta dos modelos no Drive

Procura por `<RUNS_BASE>/<model_id>_<task>_test_set/model/` (convenção dos NBs 21 / 11 / 12 / 13). LLMs e ensembles ficam de fora — exigem orquestração extra.


In [ ]:
import re
import pandas as pd

RUN_PATTERN = re.compile(r"^(bert|tfidf)_.+_(binary|multiclass)_test_set$")

discovered = []
if not RUNS_BASE.exists():
    print(f"AVISO: RUNS_BASE nao existe: {RUNS_BASE}")
else:
    for run_dir in sorted(RUNS_BASE.iterdir()):
        if not run_dir.is_dir():
            continue
        m = RUN_PATTERN.match(run_dir.name)
        if not m:
            continue
        model_dir = run_dir / "model"
        if not model_dir.is_dir():
            # Sem pesos persistidos no Drive — pular silenciosamente
            continue
        try:
            model_type = fb.detect_model_type(model_dir)
        except FileNotFoundError as exc:
            print(f"  pula {run_dir.name}: {exc}")
            continue
        task = m.group(2)
        suffix = f"_{task}_test_set"
        model_id = run_dir.name[: run_dir.name.rfind(suffix)]
        discovered.append({
            "model_id": model_id,
            "task": task,
            "model_type": model_type,
            "model_dir": str(model_dir),
            "run_dir": str(run_dir),
        })

models_df = pd.DataFrame(discovered)
print(f"\n{len(models_df)} modelos com pesos no Drive:")
display(models_df[["model_id", "task", "model_type"]] if len(models_df) else models_df)
assert len(models_df) > 0, (
    f"Nenhum modelo encontrado em {RUNS_BASE}. Verifique se os NBs 21/11/12/13 "
    "salvaram os pesos no Drive (subdir 'model/' dentro de cada run)."
)



12 modelos com pesos no Drive:


,model_id,task,model_type
0,bert_bertimbau,binary,bert
1,bert_bertimbau,multiclass,bert
2,bert_deb3rta_base,binary,bert
3,bert_deb3rta_base,multiclass,bert
4,bert_finbert_ptbr,binary,bert
5,bert_finbert_ptbr,multiclass,bert
6,tfidf_linearsvc,binary,tfidf
7,tfidf_linearsvc,multiclass,tfidf
8,tfidf_logreg,binary,tfidf
9,tfidf_logreg,multiclass,tfidf


## 5. Avaliação em loop

Para cada modelo: 1 inferência por dataset (não 1 por nível). Os helpers `evaluate_level{1,2,3}_*` consomem o mesmo array `probs` para emitir cards independentes.


In [ ]:
import time

DEFAULT_BATCH = 64
DEFAULT_MAX_LEN = 128


def run_inference(model_dir, model_type, texts):
    if model_type == "bert":
        return fb.predict_bert(
            texts, model_dir,
            batch_size=DEFAULT_BATCH, max_length=DEFAULT_MAX_LEN,
        )
    return fb.predict_tfidf(texts, model_dir)


run_log = []
for _, row in models_df.iterrows():
    model_dir = Path(row["model_dir"])
    model_id = row["model_id"]
    task = row["task"]
    model_type = row["model_type"]
    print(f"\n=== {model_id} | task={task} | type={model_type} ===")

    model_size_mb = round(compute_artifact_size_mb(model_dir), 3)
    expected_classes = 2 if task == "binary" else 8
    t0 = time.perf_counter()
    log_row = {"model_id": model_id, "task": task}

    # --- Fake.Br ---
    try:
        fb_probs, fb_classes, fb_inf_s, fb_info = run_inference(model_dir, model_type, fb_texts)
    except Exception as exc:  # noqa: BLE001
        print(f"  ERRO Fake.Br inferencia: {type(exc).__name__}: {exc}")
        continue
    if fb_probs.shape[1] != expected_classes:
        print(f"  AVISO: modelo emite {fb_probs.shape[1]} classes; esperado {expected_classes}. Pulando.")
        continue
    print(f"  Fake.Br  inf: {fb_inf_s:.1f}s, shape={fb_probs.shape}")

    common_fb = dict(
        df=fb_df, probs=fb_probs, classes=fb_classes,
        model_id=model_id, model_type=model_type,
        output_root=OOD_RUNS_BASE, inference_seconds=fb_inf_s,
        model_size_mb=model_size_mb, max_length=DEFAULT_MAX_LEN,
        n_parameters=fb_info["n_parameters"], hardware=fb_info["hardware"],
    )
    if task == "binary":
        fb.evaluate_level1_binary(**common_fb)
        fb.evaluate_level3_subgroup(**common_fb)
    else:
        fb.evaluate_level2_multiclass(**common_fb)
    log_row["fb_inference_s"] = round(fb_inf_s, 2)

    # --- FakeRecogna ---
    try:
        fr_probs, fr_classes, fr_inf_s, fr_info = run_inference(model_dir, model_type, fr_texts)
    except Exception as exc:  # noqa: BLE001
        print(f"  ERRO FakeRecogna inferencia: {type(exc).__name__}: {exc}")
        continue
    print(f"  FakeReco inf: {fr_inf_s:.1f}s, shape={fr_probs.shape}")

    common_fr = dict(
        df=fr_df, probs=fr_probs, classes=fr_classes,
        model_id=model_id, model_type=model_type,
        output_root=OOD_RUNS_BASE, inference_seconds=fr_inf_s,
        model_size_mb=model_size_mb, max_length=DEFAULT_MAX_LEN,
        n_parameters=fr_info["n_parameters"], hardware=fr_info["hardware"],
    )
    if task == "binary":
        fr.evaluate_level1_binary(**common_fr)
        fr.evaluate_level3_uol_balanced(**common_fr, task="binary")
    else:
        fr.evaluate_level2_multiclass(**common_fr)
        fr.evaluate_level3_uol_balanced(**common_fr, task="multiclass")
    log_row["fr_inference_s"] = round(fr_inf_s, 2)

    # --- PortugueseNewsDataset + RecognaSumm (somente binario, label de topico direta) ---
    if task == "binary":
        try:
            pn_probs, pn_classes, pn_inf_s, pn_info = run_inference(model_dir, model_type, pn_texts)
        except Exception as exc:  # noqa: BLE001
            print(f"  ERRO PortugueseNews inferencia: {type(exc).__name__}: {exc}")
        else:
            print(f"  PortNews inf: {pn_inf_s:.1f}s, shape={pn_probs.shape}")
            pn.evaluate_level1_binary(
                df=pn_df, probs=pn_probs, classes=pn_classes,
                model_id=model_id, model_type=model_type,
                output_root=OOD_RUNS_BASE, inference_seconds=pn_inf_s,
                model_size_mb=model_size_mb, max_length=DEFAULT_MAX_LEN,
                n_parameters=pn_info["n_parameters"], hardware=pn_info["hardware"],
                partition=PN_PARTITION,
            )
            log_row["pn_inference_s"] = round(pn_inf_s, 2)

        try:
            rs_probs, rs_classes, rs_inf_s, rs_info = run_inference(model_dir, model_type, rs_texts)
        except Exception as exc:  # noqa: BLE001
            print(f"  ERRO RecognaSumm inferencia: {type(exc).__name__}: {exc}")
        else:
            print(f"  RecognaS inf: {rs_inf_s:.1f}s, shape={rs_probs.shape}")
            rs.evaluate_level1_binary(
                df=rs_df, probs=rs_probs, classes=rs_classes,
                model_id=model_id, model_type=model_type,
                output_root=OOD_RUNS_BASE, inference_seconds=rs_inf_s,
                model_size_mb=model_size_mb, max_length=DEFAULT_MAX_LEN,
                n_parameters=rs_info["n_parameters"], hardware=rs_info["hardware"],
                partition=RS_PARTITION,
            )
            log_row["rs_inference_s"] = round(rs_inf_s, 2)

    log_row["total_s"] = round(time.perf_counter() - t0, 2)
    run_log.append(log_row)

print("\n=== RUN LOG ===")
display(pd.DataFrame(run_log))

## 6. Agregação dos cards OOD

Walk em `OOD_RUNS_BASE/*/result_card.json` filtrando por `config.domain in {fake_br_full_texts, fake_recogna_economia_uol}`.


In [ ]:
import json

OOD_DOMAINS = {
    "fake_br_full_texts",
    "fake_recogna_economia_uol",
    "portuguese_news_wikinotices",
    "recognasumm_propor2024",
}

rows = []
for card_path in sorted(OOD_RUNS_BASE.glob("*/result_card.json")):
    card = json.loads(card_path.read_text())
    domain = card.get("config", {}).get("domain")
    if domain not in OOD_DOMAINS:
        continue
    metrics = card.get("metrics", {})
    rows.append({
        "model_id": card.get("model_id"),
        "task": card.get("task"),
        "domain": domain,
        "level": card.get("config", {}).get("level"),
        "partition": card.get("config", {}).get("partition"),
        "n_eval": card.get("n_eval_samples"),
        # Binarias (se aplicaveis)
        "f1": metrics.get("f1"),
        "precision": metrics.get("precision"),
        "recall": metrics.get("recall"),
        "auc_roc": metrics.get("auc_roc"),
        "brier": metrics.get("brier"),
        "ece": metrics.get("ece"),
        "positive_prevalence": metrics.get("positive_prevalence"),
        # Multiclasse
        "macro_f1": metrics.get("macro_f1"),
        "macro_f1_present_only": metrics.get("macro_f1_present_only"),
        "weighted_f1": metrics.get("weighted_f1"),
        "accuracy": metrics.get("accuracy"),
        "card_path": str(card_path.relative_to(OOD_RUNS_BASE)),
    })

cards_df = pd.DataFrame(rows).sort_values(["domain", "task", "level", "model_id"]).reset_index(drop=True)
print(f"{len(cards_df)} cards OOD agregados\n")
display(cards_df)

### 6.1 Pivot: F1 binário por (modelo × nível)

In [ ]:
if len(cards_df):
    pivot_bin = (
        cards_df[cards_df.task == "binary"]
        .pivot_table(index="model_id", columns=["domain", "level"], values="f1")
        .round(4)
    )
    print("F1 binario:")
    display(pivot_bin)

    pivot_macro = (
        cards_df[cards_df.task == "multiclass"]
        .pivot_table(index="model_id", columns=["domain", "level"], values="macro_f1")
        .round(4)
    )
    print("\nMacro-F1 multiclasse:")
    display(pivot_macro)


F1 binario:


domain              fake_br_full_texts                                  \
level             1_binary_full_corpus 3_subgroup_fake 3_subgroup_true   
model_id                                                                 
bert_bertimbau                  0.0983          0.2169          0.0608   
bert_deb3rta_base               0.1114          0.2095          0.0709   
bert_finbert_ptbr               0.1071          0.2143          0.0714   
tfidf_linearsvc                 0.1338          0.2817          0.0808   
tfidf_logreg                    0.1150          0.2128          0.0731   
tfidf_nb                        0.1462          0.2796          0.0865   

domain            fake_recogna_economia_uol                        
level                  1_binary_full_corpus 3_uol_balanced_binary  
model_id                                                           
bert_bertimbau                       0.3876                0.6247  
bert_deb3rta_base                    0.1713                0.3027  
bert_finbert_ptbr                    0.4122                0.6926  
tfidf_linearsvc                      0.1993                0.3077  
tfidf_logreg                         0.1785                0.3110  
tfidf_nb                             0.3508                0.5760


Macro-F1 multiclasse:


domain             fake_br_full_texts fake_recogna_economia_uol  \
level             2_multiclass_mapped       2_multiclass_mapped   
model_id                                                          
bert_bertimbau                 0.1406                    0.1916   
bert_deb3rta_base              0.1386                    0.1803   
bert_finbert_ptbr              0.1423                    0.2414   
tfidf_linearsvc                0.1688                    0.1866   
tfidf_logreg                   0.1625                    0.1906   
tfidf_nb                       0.2047                    0.2602   

domain                                       
level             3_uol_balanced_multiclass  
model_id                                     
bert_bertimbau                       0.1865  
bert_deb3rta_base                    0.0965  
bert_finbert_ptbr                    0.2054  
tfidf_linearsvc                      0.1648  
tfidf_logreg                         0.1601  
tfidf_nb                             0.2001

## 7. McNemar pareado por (domain, level)

Apenas para o nível **binário** (McNemar é teste 2×2). Bonferroni aplicado automaticamente sobre `K*(K-1)/2` pares dentro de cada grupo.


In [ ]:
from collections import defaultdict

groups: dict[tuple, dict] = defaultdict(dict)
for card_path in sorted(OOD_RUNS_BASE.glob("*/result_card.json")):
    card = json.loads(card_path.read_text())
    if card.get("config", {}).get("domain") not in OOD_DOMAINS:
        continue
    if card.get("task") != "binary":
        continue
    pred_path = card_path.parent / "predictions.csv"
    if not pred_path.exists():
        continue
    pred_df = pd.read_csv(pred_path)
    if "index" not in pred_df.columns or "y_pred" not in pred_df.columns:
        continue
    key = (card["config"]["domain"], card["config"]["level"], card["task"])
    groups[key][card["model_id"]] = pred_df.set_index("index")

mcnemar_results: dict[tuple, pd.DataFrame] = {}
for (domain, level, task), preds_by_model in sorted(groups.items()):
    if len(preds_by_model) < 2:
        continue
    common_idx = None
    for df in preds_by_model.values():
        common_idx = df.index if common_idx is None else common_idx.intersection(df.index)
    aligned = {m: df.loc[common_idx, "y_pred"].to_numpy() for m, df in preds_by_model.items()}
    y_true = next(iter(preds_by_model.values())).loc[common_idx, "y_true"].to_numpy()

    pw = compute_mcnemar_pairwise(y_true, aligned)
    mcnemar_results[(domain, level, task)] = pw
    print(f"\n=== domain={domain} | level={level} | task={task} ===")
    print(f"  n={len(common_idx)}, k_modelos={len(aligned)}, n_pares={len(pw)}")
    display(pw[["method_a", "method_b", "p_value", "p_value_adjusted", "significant_after_correction"]])



=== domain=fake_br_full_texts | level=1_binary_full_corpus | task=binary ===
  n=7200, k_modelos=6, n_pares=15


,method_a,method_b,p_value,p_value_adjusted,significant_after_correction
0,bert_bertimbau,bert_deb3rta_base,0.617970,1.000000,False
1,bert_bertimbau,bert_finbert_ptbr,0.190430,1.000000,False
2,bert_bertimbau,tfidf_linearsvc,0.000000,0.000000,True
3,bert_bertimbau,tfidf_logreg,0.008151,0.122265,False
4,bert_bertimbau,tfidf_nb,0.000164,0.002463,True
5,bert_deb3rta_base,bert_finbert_ptbr,0.180194,1.000000,False
6,bert_deb3rta_base,tfidf_linearsvc,0.000000,0.000000,True
7,bert_deb3rta_base,tfidf_logreg,0.002073,0.031093,True
8,bert_deb3rta_base,tfidf_nb,0.000054,0.000816,True
9,bert_finbert_ptbr,tfidf_linearsvc,0.000000,0.000001,True



=== domain=fake_br_full_texts | level=3_subgroup_fake | task=binary ===
  n=3600, k_modelos=6, n_pares=15


,method_a,method_b,p_value,p_value_adjusted,significant_after_correction
0,bert_bertimbau,bert_deb3rta_base,0.026716,0.400735,False
1,bert_bertimbau,bert_finbert_ptbr,0.852684,1.000000,False
2,bert_bertimbau,tfidf_linearsvc,0.052204,0.783055,False
3,bert_bertimbau,tfidf_logreg,0.241317,1.000000,False
4,bert_bertimbau,tfidf_nb,0.799495,1.000000,False
5,bert_deb3rta_base,bert_finbert_ptbr,0.037813,0.567189,False
6,bert_deb3rta_base,tfidf_linearsvc,0.000048,0.000724,True
7,bert_deb3rta_base,tfidf_logreg,0.264288,1.000000,False
8,bert_deb3rta_base,tfidf_nb,0.059346,0.890197,False
9,bert_finbert_ptbr,tfidf_linearsvc,0.046945,0.704171,False



=== domain=fake_br_full_texts | level=3_subgroup_true | task=binary ===
  n=3600, k_modelos=6, n_pares=15


,method_a,method_b,p_value,p_value_adjusted,significant_after_correction
0,bert_bertimbau,bert_deb3rta_base,0.336515,1.000000,False
1,bert_bertimbau,bert_finbert_ptbr,0.079616,1.000000,False
2,bert_bertimbau,tfidf_linearsvc,0.000000,0.000000,True
3,bert_bertimbau,tfidf_logreg,0.000044,0.000660,True
4,bert_bertimbau,tfidf_nb,0.000004,0.000053,True
5,bert_deb3rta_base,bert_finbert_ptbr,0.862829,1.000000,False
6,bert_deb3rta_base,tfidf_linearsvc,0.000001,0.000019,True
7,bert_deb3rta_base,tfidf_logreg,0.002700,0.040497,True
8,bert_deb3rta_base,tfidf_nb,0.000328,0.004922,True
9,bert_finbert_ptbr,tfidf_linearsvc,0.000000,0.000003,True



=== domain=fake_recogna_economia_uol | level=1_binary_full_corpus | task=binary ===
  n=11872, k_modelos=6, n_pares=15


,method_a,method_b,p_value,p_value_adjusted,significant_after_correction
0,bert_bertimbau,bert_deb3rta_base,0.000070,0.001055,True
1,bert_bertimbau,bert_finbert_ptbr,0.088339,1.000000,False
2,bert_bertimbau,tfidf_linearsvc,0.607646,1.000000,False
3,bert_bertimbau,tfidf_logreg,0.000439,0.006585,True
4,bert_bertimbau,tfidf_nb,0.438578,1.000000,False
5,bert_deb3rta_base,bert_finbert_ptbr,0.008010,0.120149,False
6,bert_deb3rta_base,tfidf_linearsvc,0.000110,0.001644,True
7,bert_deb3rta_base,tfidf_logreg,0.698311,1.000000,False
8,bert_deb3rta_base,tfidf_nb,0.001117,0.016754,True
9,bert_finbert_ptbr,tfidf_linearsvc,0.476033,1.000000,False



=== domain=fake_recogna_economia_uol | level=3_uol_balanced_binary | task=binary ===
  n=624, k_modelos=6, n_pares=15


,method_a,method_b,p_value,p_value_adjusted,significant_after_correction
0,bert_bertimbau,bert_deb3rta_base,0.000000,0.000000,True
1,bert_bertimbau,bert_finbert_ptbr,0.001279,0.019185,True
2,bert_bertimbau,tfidf_linearsvc,0.000000,0.000000,True
3,bert_bertimbau,tfidf_logreg,0.000000,0.000000,True
4,bert_bertimbau,tfidf_nb,0.112924,1.000000,False
5,bert_deb3rta_base,bert_finbert_ptbr,0.000000,0.000000,True
6,bert_deb3rta_base,tfidf_linearsvc,0.725496,1.000000,False
7,bert_deb3rta_base,tfidf_logreg,0.905530,1.000000,False
8,bert_deb3rta_base,tfidf_nb,0.000000,0.000000,True
9,bert_finbert_ptbr,tfidf_linearsvc,0.000000,0.000000,True


## 8. Export para `<DRIVE>/reports/ood_evaluation/`

In [ ]:
REPORT_DIR = OOD_RUNS_BASE.parent / "reports" / "ood_evaluation"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

cards_df.to_csv(REPORT_DIR / "ood_cards_table.csv", index=False)
print(f"Tabela de cards: {REPORT_DIR / 'ood_cards_table.csv'}")

if len(cards_df):
    if (cards_df.task == "binary").any():
        cards_df[cards_df.task == "binary"].pivot_table(
            index="model_id", columns=["domain", "level"], values="f1",
        ).round(4).to_csv(REPORT_DIR / "ood_pivot_binary_f1.csv")
        print(f"Pivot F1 binario: {REPORT_DIR / 'ood_pivot_binary_f1.csv'}")
    if (cards_df.task == "multiclass").any():
        cards_df[cards_df.task == "multiclass"].pivot_table(
            index="model_id", columns=["domain", "level"], values="macro_f1",
        ).round(4).to_csv(REPORT_DIR / "ood_pivot_multiclass_macrof1.csv")
        print(f"Pivot macro-F1 multi: {REPORT_DIR / 'ood_pivot_multiclass_macrof1.csv'}")

for (domain, level, task), pw in mcnemar_results.items():
    fname = f"mcnemar_{domain}_{level}_{task}.csv"
    pw.to_csv(REPORT_DIR / fname, index=False)
    print(f"McNemar: {REPORT_DIR / fname}")

print(f"\nDestino: {REPORT_DIR}")


Tabela de cards: /content/drive/MyDrive/economy-classifier/reports/ood_evaluation/ood_cards_table.csv
Pivot F1 binario: /content/drive/MyDrive/economy-classifier/reports/ood_evaluation/ood_pivot_binary_f1.csv
Pivot macro-F1 multi: /content/drive/MyDrive/economy-classifier/reports/ood_evaluation/ood_pivot_multiclass_macrof1.csv
McNemar: /content/drive/MyDrive/economy-classifier/reports/ood_evaluation/mcnemar_fake_br_full_texts_1_binary_full_corpus_binary.csv
McNemar: /content/drive/MyDrive/economy-classifier/reports/ood_evaluation/mcnemar_fake_br_full_texts_3_subgroup_fake_binary.csv
McNemar: /content/drive/MyDrive/economy-classifier/reports/ood_evaluation/mcnemar_fake_br_full_texts_3_subgroup_true_binary.csv
McNemar: /content/drive/MyDrive/economy-classifier/reports/ood_evaluation/mcnemar_fake_recogna_economia_uol_1_binary_full_corpus_binary.csv
McNemar: /content/drive/MyDrive/economy-classifier/reports/ood_evaluation/mcnemar_fake_recogna_economia_uol_3_uol_balanced_binary_binary.csv

